# Primera Red Neuronal con PyTorch

Hasta ahora hemos construido redes neuronales manualmente con NumPy para entender qué ocurre dentro del modelo.

En esta sección comenzaremos a utilizar **PyTorch**, una de las librerías más utilizadas para Deep Learning.

La meta no es reemplazar lo aprendido anteriormente, sino conectar cada concepto con su implementación real:

```text
NumPy / Matemática        PyTorch
-------------------       ------------------
weights y bias        →   nn.Linear
ReLU                  →   nn.ReLU
forward pass          →   forward()
loss                  →   loss function
backpropagation       →   loss.backward()
gradient descent      →   optimizer.step()
```

Trabajaremos nuevamente con el dataset **Iris**, porque permite concentrarnos en la red neuronal sin complicar demasiado los datos.

## Objetivos

Al finalizar podrás:

- convertir datos científicos a tensores;
- crear una red con `nn.Module`;
- interpretar `nn.Linear`;
- realizar un forward pass;
- entender las dimensiones de entrada y salida;
- inspeccionar los parámetros de una red;
- obtener predicciones iniciales antes del entrenamiento.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn


## 1. Cargamos el dataset Iris


In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

print("X.shape =", X.shape)
print("y.shape =", y.shape)
print("Clases:", iris.target_names)


Tenemos:

```text
150 samples
4 features
3 clases
```

Por lo tanto, una red para este problema debe tener:

```text
4 inputs
...
3 outputs
```

La parte intermedia la podemos diseñar nosotros.


## 2. Training y test sets

Antes de entrenar un modelo, separamos los datos.

Utilizaremos:

```text
80% training
20% test
```


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Test:", X_test.shape)


### Pregunta

¿Por qué no debemos evaluar el modelo únicamente con los mismos datos que utilizamos para entrenarlo?

<details>
<summary><strong>Pista</strong></summary>

Queremos saber si el modelo puede funcionar con observaciones que no vio durante el entrenamiento.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

Porque un modelo puede aprender muy bien los ejemplos de entrenamiento sin necesariamente **generalizar** a observaciones nuevas.

El test set nos permite evaluar el modelo con datos que no participaron en el proceso de aprendizaje.

</details>


## 3. Estandarización

Las features pueden tener escalas distintas.

Una transformación común es:

\[
x' = \frac{x-\mu}{\sigma}
\]

donde:

- \(\mu\) es la media;
- \(\sigma\) es la desviación estándar.

Esto hace que las variables tengan escalas comparables.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Media aproximada:", X_train_scaled.mean(axis=0))
print("Std aproximada:", X_train_scaled.std(axis=0))


Observa que usamos:

```python
fit_transform(X_train)
```

pero solamente:

```python
transform(X_test)
```

### Pregunta

¿Por qué no hacemos `fit_transform(X_test)`?

<details>
<summary><strong>Pista</strong></summary>

El test set debe comportarse como información nueva que el modelo nunca utilizó.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

Porque ajustar el scaler usando el test set permitiría que información del conjunto de prueba influyera indirectamente en el proceso.

Eso se conoce como **data leakage**.

</details>


## 4. De NumPy a PyTorch tensors

PyTorch trabaja con objetos llamados **tensors**.

Son conceptualmente similares a arrays de NumPy, pero están diseñados para:

- automatic differentiation;
- GPUs;
- redes neuronales.


In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print(X_train_t.shape)
print(y_train_t.shape)
print(X_train_t.dtype)
print(y_train_t.dtype)


## 5. Diseñando nuestra primera red

Crearemos:

```text
4 → 8 → 3
```

Interpretación:

```text
4 scientific features
        ↓
8 hidden neurons
        ↓
3 output classes
```

En PyTorch:

```python
nn.Linear(4, 8)
nn.ReLU()
nn.Linear(8, 3)
```


In [ ]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

print(model)


### Conexión con lo aprendido

La primera capa:

```python
nn.Linear(4, 8)
```

implementa:

\[
\mathbf{z}_1 = W_1\mathbf{x}+b_1.
\]

Después:

```python
nn.ReLU()
```

aplica:

\[
\mathbf{a}_1=\mathrm{ReLU}(\mathbf{z}_1).
\]

Finalmente:

```python
nn.Linear(8, 3)
```

produce tres valores de salida.


## 6. ¿Cuántos parámetros tiene la red?

Antes de ejecutar, calcula manualmente:

```text
4 → 8 → 3
```

<details>
<summary><strong>Pista</strong></summary>

Primera capa:

\[
4\times8+8
\]

Segunda capa:

\[
8\times3+3
\]

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

Primera capa:

\[
4\times8+8=40
\]

Segunda capa:

\[
8\times3+3=27
\]

Total:

\[
67
\]

parámetros entrenables.

</details>


In [ ]:
total_params = sum(p.numel() for p in model.parameters())

print("Total parameters:", total_params)


## 7. Forward pass

Tomemos cinco flores y pasémoslas por la red.


In [ ]:
with torch.no_grad():
    logits = model(X_test_t[:5])

print(logits)
print("shape =", logits.shape)


La salida tiene shape:

```text
(5, 3)
```

porque tenemos:

```text
5 samples
3 output values por sample
```

Estos valores se llaman **logits**.

Todavía no son probabilidades.


## 8. Softmax

Podemos convertir logits en valores entre 0 y 1 usando:

\[
p_i =
\frac{e^{z_i}}
{\sum_j e^{z_j}}.
\]

Esto es **Softmax**.


In [ ]:
softmax = nn.Softmax(dim=1)

with torch.no_grad():
    probabilities = softmax(logits)

print(probabilities)
print("Suma por sample:", probabilities.sum(dim=1))


### Pregunta

¿Por qué cada fila suma aproximadamente 1?

<details>
<summary><strong>Mostrar solución</strong></summary>

Porque Softmax normaliza los logits para producir una distribución sobre las clases.

Podemos interpretarlos como probabilidades relativas entre las tres categorías.

</details>


## 9. Predicción inicial

La clase predicha será la posición con el valor más alto.


In [ ]:
with torch.no_grad():
    predictions = torch.argmax(logits, dim=1)

print("Predicciones:", predictions)
print("Valores reales:", y_test_t[:5])


Estas predicciones probablemente no serán muy buenas todavía.

¿Por qué?

Porque los weights y biases fueron inicializados automáticamente, pero **todavía no han sido entrenados**.

La red existe.

El forward pass funciona.

Pero aún no ha aprendido.


## 10. Usando una clase `nn.Module`

También podemos definir la misma arquitectura de forma explícita.

Esto será útil cuando las redes sean más complejas.


In [ ]:
class IrisNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(8, 3)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


model_class = IrisNetwork()

print(model_class)


### Pregunta

¿Dónde ocurre el forward pass?

<details>
<summary><strong>Mostrar solución</strong></summary>

En:

```python
def forward(self, x):
```

PyTorch llama este método cuando hacemos:

```python
model(x)
```

</details>


# Para recordar

Ya tenemos una red real en PyTorch:

```text
4 features
    ↓
Linear(4,8)
    ↓
ReLU
    ↓
Linear(8,3)
    ↓
logits
```

Pero todavía falta el paso fundamental:

> **entrenar los parámetros utilizando loss, backpropagation y gradient descent.**

Eso lo haremos en el siguiente notebook.


## Recursos

- [PyTorch — Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- [PyTorch — Tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
